# Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.optimize as opt
import time
import warnings
warnings.filterwarnings('ignore')

from tqdm.auto import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # torch.device('mps')
SEED = 590704
np.random.seed(SEED)

# Pre-Pare Dataset
* test size = 0.33
* train & test set count : 50
    * for -> SEED + 1

In [ ]:
data_path = "t_distribution_xy_data.csv" # 데이터 경로 정의
data = pd.read_csv(data_path) # 데이터프레임으로 데이터 호출

In [ ]:
data.head(3)

In [ ]:
# Data Pre-processing
# id 컬럼에 대해서 0 ~ unique한 id의 갯수까지 로 Label Encoding 수행
label_encoder = LabelEncoder()
data['id'] = label_encoder.fit_transform(data['id'])
clusters = sorted(list(set(list(data['id'].values)))) # 유니크한 cluster(id)의 리스트 정의

In [ ]:
# Torch Dataset 생성 클래스
class SIMDataset(Dataset):
    def __init__(self, data, targets, cluster_ids):
        self.data = data.astype(np.float32)
        self.targets = targets.astype(np.float32)
        self.cluster_ids = cluster_ids

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx], self.cluster_ids[idx]

In [ ]:
x_col = ['id', 'x'] # 종속 변수 컬럼명
y_col = ["y"] # 독립 변수 컬럼명

train_grp = []
test_grp = []

# 데이터셋 쌍 50개 준비
for i in range(50):
    seed = SEED + i
    # 결과를 저장할 리스트
    train_list = []
    test_list = []

    train, test = train_test_split(data, test_size=0.333, random_state=seed, stratify=data['id'])
    
    # 데이터셋 생성
    train_dataset = SIMDataset(train[x_col].values, train[y_col].values, train["id"].values)
    test_dataset = SIMDataset(test[x_col].values, test[y_col].values, test["id"].values)

    batch_size = 32 # 데이터 배치사이즈 정의
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    train_grp.append(train_loader)
    test_grp.append(test_loader)

# Model

In [ ]:
class tMeNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(tMeNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, input_dim)
        self.fc5 = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

    # feature map으로 Z 생성하기 위한 함수
    def get_feature_map(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        feature_map = torch.relu(self.fc4(x))
        return feature_map

# ECML Algorithm

### E-Step : 잠재 변수와 관련된 기대치 계산

In [ ]:
def e_step(X, y, beta_hat, psi_hat, sigma2_hat, nu_hat, cluster_ids, device):
    """
    - X, Z : X와 Z를 동일한 데이터로 가정한다.
    - y : 타겟 데이터
    - beta_hat, psi_hat, sigma_squared_hat, nu_hat: 현재 추정치
    - cluster_ids, clusters: 클러스터 아이디와 클러스터 리스트
    - device: 계산을 수행할 디바이스 (예: 'cuda', 'cpu')
    """
    n, d = X.shape  # n: batch size, d: feature dimension
    clusters = torch.unique(cluster_ids).tolist()  # Get unique cluster IDs
    
    # Initialize storage for results
    b_hat = torch.zeros((n, d, y.shape[1]), device=device)
    tau_hat = torch.zeros(n, device=device)
    Omega_hat = torch.zeros((n, d, d), device=device)

    for cluster_id in clusters:
        indices = (cluster_ids == cluster_id).nonzero(as_tuple=True)[0]
        if indices.numel() == 0:
            continue
        
        # Extract data for the current cluster
        X_cluster = X[indices]
        y_cluster = y[indices]
        
        # Handling of beta_hat; Assuming beta_hat is global for simplicity. Adjust if necessary.
        beta_cluster = beta_hat.to(device)#.unsqueeze(-1)
        
        # Ensure psi_hat, sigma2_hat, and nu_hat are accessed correctly for the current cluster
        # Handling psi_hat based on its dimensionality
        regularization_term = 1e-3  # 또는 이 값을 조정하여 실험
        psi_hat_reg = psi_hat[cluster_id].to(device) + regularization_term * torch.eye(psi_hat[cluster_id].size(0), device=device)
        psi_inv_cluster = torch.linalg.pinv(psi_hat_reg)

        # psi_inv_cluster = torch.linalg.inv(psi_hat[cluster_id].to(device)) if psi_hat.ndim > 1 else torch.linalg.inv(psi_hat.to(device))
        sigma2_inv_cluster = 1.0 / sigma2_hat[cluster_id].to(device)
        nu_cluster = nu_hat[cluster_id].to(device)
        
        # Compute A, Omega, b_hat, and tau_hat for the current cluster
        A_cluster = psi_inv_cluster + sigma2_inv_cluster * X_cluster.T @ X_cluster
        Omega_cluster = torch.linalg.pinv(A_cluster)
        # U, S, V = torch.linalg.svd(A_cluster, full_matrices=False)
        # S_inv = torch.diag(1.0 / S)
        # Omega_cluster = V @ S_inv @ U.T
        
        b_cluster = Omega_cluster @ (sigma2_inv_cluster * X_cluster.T @ (y_cluster - X_cluster @ beta_cluster))
        delta_squared_cluster = (b_cluster.T @ psi_inv_cluster @ b_cluster +
                                 sigma2_inv_cluster * (y_cluster - X_cluster @ b_cluster).T @
                                 (y_cluster - X_cluster @ b_cluster)).squeeze()
        
        tau_cluster = (nu_cluster + X_cluster.shape[0]) / (nu_cluster + delta_squared_cluster)
        
        # Assign computed values back to the appropriate indices for the entire batch
        for i, index in enumerate(indices):
            b_hat[index] = b_cluster  # Directly assigning without reshaping as b_cluster matches the expected shape

        tau_hat[indices] = tau_cluster.mean()  # Assuming a scalar value for simplicity
        tau_hat[indices] = torch.clamp(tau_hat[indices], min=1e-6)
        
        for idx in indices:
            Omega_hat[idx] = Omega_cluster  # Assigning Omega_cluster to each corresponding index
        
    return b_hat, tau_hat, Omega_hat

### CM-Step 1 : $\beta$ 파라미터 업데이트

In [ ]:
def cm_step_1(X, y, tau, beta_hat, Z, sigma2_hat, cluster_ids, device):
    """
    CM-step 1: Update beta_hat considering the current estimates and observations.

    Parameters:
    - X: The feature matrix of shape [n, d].
    - y: The target matrix of shape [n, k], where k is the number of targets per observation.
    - tau: The weights for each observation, shape [n].
    - beta_hat: The current estimate of beta, shape [d, k].
    - Z: The matrix used in the model, assumed to be the same as X for simplification.
    - sigma2_hat: The current estimate of sigma squared, not directly used here for simplification.
    - cluster_ids: The cluster IDs for each observation, shape [n].
    - device: The computing device ('cuda' or 'cpu').

    Returns:
    - Updated beta_hat.
    """
    n, d = X.shape
    k = y.shape[1]  # Number of targets per observation

    # Ensure everything is on the correct device
    X, y, tau = X.to(device), y.to(device), tau.to(device)
    beta_hat, Z = beta_hat.to(device), Z.to(device)

    # Initialize accumulators
    weighted_X = torch.zeros((d, d), device=device)
    weighted_Xy = torch.zeros((d, k), device=device)

    for i in range(n):
        X_i = X[i].unsqueeze(0)  # Reshape to [1, d] for matrix operations
        y_i = y[i].unsqueeze(0)  # Reshape to [1, k]
        tau_i = tau[i]
        
        # Update the accumulators
        weighted_X += tau_i * X_i.T @ X_i
        weighted_Xy += tau_i * X_i.T @ y_i  # Directly use y_i

    # Solve for beta_hat using the normal equation
    beta_hat_updated = torch.linalg.solve(weighted_X, weighted_Xy)
    return beta_hat_updated

### CM-Step 2 : $\sigma^2$ 파라미터 업데이트

In [ ]:
def cm_step_2(X, y, beta_hat, tau, sigma_squared_hat, cluster_ids, clusters, device):
    """
    CM-step 2: Update sigma squared (sigma2_hat) for each cluster considering the fixed beta_hat.

    Parameters:
    - X: The feature matrix of shape [n, d].
    - y: The target matrix of shape [n, k], where k is the number of targets per observation.
    - beta_hat: The current estimate of beta, shape [d, k].
    - tau: The weights for each observation, shape [n].
    - cluster_ids: Array of cluster IDs for each observation, shape [n].
    - clusters: Unique list of cluster IDs.
    - device: The computing device ('cuda' or 'cpu').

    Returns:
    - Updated sigma squared (sigma2_hat) for each cluster.
    """
    for cluster in clusters:
        # Mask to select data points belonging to the current cluster
        cluster_mask = (cluster_ids == cluster)
        if cluster_mask.sum() > 0: 
            # Select data for the current cluster
            X_cluster, y_cluster, tau_cluster = X[cluster_mask], y[cluster_mask], tau[cluster_mask]
            # Compute residuals for the cluster
            residuals_cluster = y_cluster - X_cluster @ beta_hat  # Shape [n_cluster, k]
            # Apply weights to the squared residuals
            weighted_residuals_cluster = tau_cluster.unsqueeze(-1) * (residuals_cluster ** 2)
            # Sum the weighted residuals and normalize by the number of observations times the number of targets in the cluster
            n_cluster, k = y_cluster.shape
            sigma_squared_hat[cluster] = weighted_residuals_cluster.sum() / (n_cluster * k)

    return sigma_squared_hat

### CM Step 3 : $\psi$ 파라미터 업데이트

In [ ]:
def cm_step_3(b_hat, Omega_hat, tau_hat, psi_hat, cluster_ids, device):
    """
    Modified CM-step 3 to include tau_hat weights in the psi_hat update.

    Parameters:
    - b_hat: Estimated random effects for each observation [n, d].
    - Omega_hat: Estimated covariance of random effects for each observation [n, d, d].
    - tau_hat: Weights for each observation, reflecting their relative importance [n].
    - device: The computing device ('cuda' or 'cpu').

    Returns:
    - Updated psi_hat.
    """
    clusters = torch.unique(cluster_ids).tolist()  # Get unique cluster IDs
    d = psi_hat.size(1)  # Assuming psi_hat is [num_clusters, d, d]

    psi_hat_updated = torch.zeros_like(psi_hat)

    for cluster_id in clusters:
        indices = (cluster_ids == cluster_id).nonzero(as_tuple=True)[0]
        if indices.numel() == 0:
            continue

        n_i = indices.size(0)
        psi_component_cluster = torch.zeros((d, d), device=device)

        for i in indices:
            psi_component = tau_hat[i] * Omega_hat[i] + torch.outer(b_hat[i,:,0], b_hat[i,:,0])
            psi_component_cluster += psi_component

        psi_component_cluster /= n_i
        psi_component_cluster = 0.5 * (psi_component_cluster + psi_component_cluster.T)  # Symmetrize
        psi_component_cluster += 1e-6 * torch.eye(d, device=device)
        psi_hat_updated[cluster_id] = psi_component_cluster

    return psi_hat_updated

### CML-Step 4 : $\nu$ 파라미터 업데이트 (자유도)

In [ ]:
def calculate_delta_squared(beta_hat, sigma2_hat, X, Z, y):
    residuals = y - X @ beta_hat
    delta_squared = (residuals ** 2).sum() / (sigma2_hat + 1)
    delta_squared = torch.clamp(delta_squared, min=1e-2)
    return delta_squared

def objective_function(nu, delta_squared, n_i):
    term1 = torch.lgamma((nu + n_i) / 2.0) - torch.lgamma(nu / 2.0)
    term2 = (nu / 2.0) * torch.log(nu)
    term3 = -((nu + n_i) / 2.0) * torch.log(nu + delta_squared)  # 안전한 계산
    return -(term1 + term2 + term3).sum()

def cml_step_4(X, y, Z, beta_hat, psi_hat, sigma2_hat, cluster_ids, nu_hat, device):
    clusters = torch.unique(cluster_ids)
    optimizer = optim.Adam([nu_hat], lr=0.001)
    
    for i, cluster in enumerate(clusters):
        mask = (cluster_ids == cluster)
        if not mask.any():
            continue  # 데이터가 없는 클러스터는 넘어갑니다.
        X_cluster, y_cluster, Z_cluster = X[mask], y[mask], Z[mask]
        sigma2_cluster = sigma2_hat[i]
        
        delta_squared = calculate_delta_squared(beta_hat, sigma2_cluster, X_cluster, Z_cluster, y_cluster)
        
        for iteration in range(10):
            optimizer.zero_grad()
            loss = objective_function(nu_hat[i], delta_squared, X_cluster.size(0))
            
            if torch.isnan(loss).any():
                continue
            loss.backward(retain_graph=True)
            optimizer.step()
            
            with torch.no_grad():
                nu_hat[i] = torch.clamp(nu_hat[i], min=1e-3).detach().clone()
    return nu_hat

# Model Train

### Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0, checkpoint_path='./checkpoint.pt'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.checkpoint_path = checkpoint_path

    def __call__(self, val_loss, model):
        if np.isnan(val_loss):
            print("Validation loss is NaN. Skipping model save and increasing early stop counter.")
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), self.checkpoint_path)
        self.val_loss_min = val_loss

### Negative Log-Likelihood Loss

In [ ]:
def loss_function(X, y, Z, psi_hat, sigma_squared_hat, b_hat, tau_hat, nu, group_indices, k, device):
    # 초기 항목들을 0으로 설정
    L1_term = torch.tensor(0.0, device=device)
    L2_term = torch.tensor(0.0, device=device)
    L3_term = torch.tensor(0.0, device=device)
    constant_term = torch.tensor(0.0, device=device)
    
    # 각 데이터 포인트가 속한 클러스터의 ID를 유일한 값으로 추출
    unique_clusters = torch.unique(group_indices)
    
    # 각 클러스터별로 반복하여 계산을 수행
    for j, cluster in enumerate(unique_clusters):
        # 현재 클러스터에 속한 데이터 포인트들의 인덱스를 추출
        indices = (group_indices == cluster).nonzero(as_tuple=True)[0]
        if len(indices) == 0:
            continue  # 클러스터에 데이터가 없는 경우 건너뛰기
        X_cluster = X[indices]
        y_cluster = y[indices]
        Z_cluster = Z[indices]
        b_i = b_hat[indices].squeeze(-1)
        n_i = X_cluster.size(0)  # 현재 클러스터에 속한 데이터 포인트의 수

        # sigma_squared_hat의 안전한 값 확보 (음수 방지)
        safe_sigma_squared = sigma_squared_hat[j].clamp(min=1e-3)
        tau_value = tau_hat[indices].mean()  # 현재 클러스터에 대한 tau_hat의 평균값

        # L1 항 계산
        y_i_minus_Z_b = y_cluster - Z_cluster @ b_i.unsqueeze(-1)
        trace_term = torch.einsum('bij, bji->b', [y_i_minus_Z_b, y_i_minus_Z_b.transpose(1, 2)])
        sigma_log = torch.log(safe_sigma_squared)
        if torch.isnan(sigma_log) or torch.isinf(sigma_log):
            sigma_log = torch.tensor(1.0, device=device)
        
        L1_term += -0.5 * n_i * sigma_log
        L1_term -= 0.5 * torch.sum(tau_value / safe_sigma_squared * trace_term)

        # L2 항 계산
        psi_hat_j = psi_hat[j]
        outer_product_sum = torch.zeros_like(psi_hat_j)
        for i in indices:
            outer_product = torch.outer(b_hat[i,:,0], b_hat[i,:,0])
            outer_product_sum += tau_hat[i] * outer_product  # tau에 의해 가중치가 적용된 외적의 합

        try:
            psi_hat_inv = torch.linalg.pinv(psi_hat_j)
            logdet_result = torch.logdet(psi_hat_j)
        except RuntimeError as e:
            psi_hat_inv = torch.eye(psi_hat_j.size(0), device=device)  # 대체 값
            logdet_result = torch.tensor(1.0, device=device)  # 로그 행렬식의 대체 값

        # psi_hat_inv = torch.linalg.pinv(psi_hat_j)  # 역행렬 계산
        trace_value = torch.trace(psi_hat_inv @ outer_product_sum)  # trace 계산
        logdet_result = torch.logdet(psi_hat_j)
        if torch.isnan(logdet_result) or torch.isinf(logdet_result):
            logdet_result = torch.tensor(1.0, device=device)

        L2_term += -0.5 * len(indices) * logdet_result - 0.5 * trace_value
        L2_term_threshold = 1e5
        L2_term = min(L2_term, L2_term_threshold)
    
    # L3 항 계산
    # 각 클러스터별로 nu와 tau에 대한 계산을 수행
    tau_hat_mapping = {cluster.item(): tau_hat[j] for j, cluster in enumerate(unique_clusters)}
    for j in range(1, k + 1):
        nu_j_safe = nu[j - 1].clamp(min=1e-8)
        for i, group in enumerate(group_indices):
            if group == j:
                tau_i_safe = tau_hat_mapping[group.item()].clamp(min=1e-3)
                L3_term += (nu_j_safe / 2) * (torch.log(nu_j_safe / 2 + 1e-3) + torch.log(tau_i_safe + 1e-3) - tau_i_safe) - torch.log(tau_i_safe + 1e-3) - torch.lgamma(nu_j_safe / 2)
                if torch.isnan(L3_term) or torch.isinf(L3_term):
                    L3_term = torch.tensor(1.0, device=device)
    # 최종 손실 계산
    # print(f"L1_tetm : {L1_term}")
    # print(f"L2_term : {L2_term}")
    # print(f"L3_term : {L3_term}")
    if torch.isnan(L1_term) or torch.isinf(L1_term):
        L1_term = torch.tensor(1.0, device=device)
    if torch.isnan(L2_term) or torch.isinf(L2_term):
        L2_term = torch.tensor(1.0, device=device)
    log_likelihood_value = L1_term + L2_term + L3_term + constant_term
    return -log_likelihood_value

In [ ]:
# MRAE
def mean_relative_absolute_error(y_real, y_pred):
    """
    Calculate Mean Relative Absolute Error (MRAE).

    Parameters:
    actual (list): List of actual values.
    predicted (list): List of predicted values.

    Returns:
    float: MRAE value.
    """
    # errors = [abs(a - p) / max(1e-8, abs(a)) for a, p in zip(y_real, y_pred)]
    # mrae = sum(errors) / len(errors)
    if not isinstance(y_pred, np.ndarray):
        y_pred = y_pred.detach().numpy()

    # y_real를 numpy 배열로 변환
    y_real = np.array(y_real)
    
    # y_real의 길이를 계산
    n = y_real.shape[0]

    # y_real을 (n, 1) 형태로 리쉐이핑하여 MRAE 계산
    mrae = np.mean(np.abs(y_real.reshape(n, 1) - y_pred) / y_real.reshape(n, 1)) * 100

    return mrae

# MRPE
def mean_relative_prediction_error(y_real, y_pred):
    """
    Calculate Mean Relative Prediction Error (MRPE).

    Parameters:
    actual (list): List of actual values.
    predicted (list): List of predicted values.

    Returns:
    float: MRPE value.
    """
    
    # errors = [(a - p) / max(1e-8, abs(a)) for a, p in zip(y_real, y_pred)]
    # mrpe = sum(errors) / len(errors)
    # y_pred가 Tensor 타입인 경우 numpy 배열로 변환
    if not isinstance(y_pred, np.ndarray):
        y_pred = y_pred.detach().numpy()

    # y_test를 numpy 배열로 변환
    y_real = np.array(y_real)
    
    # y_test의 길이를 계산
    n = y_real.shape[0]

    # y_test를 (n, 1) 형태로 리쉐이핑하여 MRPE 계산
    mrpe = np.mean((y_real.reshape(n, 1) - y_pred) / y_real.reshape(n, 1)) * 100

    return mrpe

### Train Loop

In [ ]:
def train_model(model, input_dim, pretrain, pretrain_epochs, epochs, train_loader, clusters, patience, verbose, device, use_early_stopping):
    """
    모델을 학습하는 주요 루프입니다.
    """
    if use_early_stopping:
        early_stopping = EarlyStopping(patience=patience, verbose=verbose)
    
    optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999), weight_decay=0.01)

    # Pre-Training
    if pretrain:
        loss_fn = nn.MSELoss()
        for epoch in range(pretrain_epochs):
            model.train()
            pretrain_loss = 0
            for data, target, _ in train_loader:
                optimizer.zero_grad()
                output = model(data)
                loss = loss_fn(output, target)
                loss.backward()
                optimizer.step()
                pretrain_loss += loss.item()
    
    # Main Training
    # Initialize beta_hat, psi_hat, sigma2_hat, nu_hat
    # These should be initialized according to your model's architecture and data dimension
    # For demonstration, initializing them with placeholder values
    beta_hat = torch.randn(input_dim, 1, device=device)  # Placeholder initialization
    psi_hat = torch.stack([torch.eye(input_dim, device=device) for _ in range(len(clusters))]) # Placeholder initialization
    sigma2_hat = torch.ones(len(clusters), device=device)  # One sigma^2 value per cluster
    nu_hat = torch.ones(len(clusters), device=device, requires_grad=True)      # One nu value per cluster
    
    mrae_loss = []
    mrpe_loss = []
    for epoch in tqdm(range(epochs)):
        st_time = time.time()
        for X_batch, Y_batch, cluster_id in train_loader:
            optimizer.zero_grad()
            # get batched data
            X_batch, Y_batch, cluster_ids_batch = X_batch.to(device), Y_batch.to(device), cluster_id.to(device)
            # 모델의 피처 맵 생성
            Z_batch = model.get_feature_map(X_batch)
            
            # ECML 알고리즘 수행 전 파라미터 복사 및 분리
            beta_hat_copy = beta_hat.clone().detach()
            psi_hat_copy = psi_hat.clone().detach()
            sigma2_hat_copy = sigma2_hat.clone().detach()
            
            
            # ECML 알고리즘 수행 (복사본 파라미터 사용)
            b_hat, tau_hat, Omega_hat = e_step(X_batch, Y_batch, beta_hat_copy, psi_hat_copy, sigma2_hat_copy, nu_hat, cluster_ids_batch, device)
            beta_hat = cm_step_1(X_batch, Y_batch, tau_hat, Z_batch, beta_hat_copy, sigma2_hat_copy, cluster_ids_batch, device)
            sigma2_hat = cm_step_2(X_batch, Y_batch, beta_hat, tau_hat, sigma2_hat_copy, cluster_ids_batch, clusters, device)
            psi_hat = cm_step_3(b_hat, Omega_hat, tau_hat, psi_hat_copy, cluster_ids_batch, device)
            nu_hat = cml_step_4(X_batch, Y_batch, Z_batch, beta_hat, psi_hat, sigma2_hat, cluster_ids_batch, nu_hat, device)

            # Calculate Loss
            y_pred = model(X_batch).clone()
            mrae = mean_relative_absolute_error(Y_batch, y_pred)
            mrae_loss.append(mrae)
            mrpe = mean_relative_prediction_error(Y_batch, y_pred)
            mrpe_loss.append(mrpe)
            loss = loss_function(X_batch, Y_batch, Z_batch, psi_hat, sigma2_hat, b_hat, tau_hat, nu_hat, cluster_ids_batch, len(clusters), device)
            # backpropagation & optimization
            # print(f"loss : {loss}")
            try:
                loss.backward()
            except:
                pass
                # print("loss Exception")
            optimizer.step()
        
        ed_time = time.time()
        mrae_value = sum(mrae_loss) / len(mrae_loss)
        mrpe_value = sum(mrpe_loss) / len(mrpe_loss)
        print(f"Epoch {epoch + 1}, LL Loss : {loss.item()}, MRAE : {mrae_value}, MRPE : {mrpe_value}, Time : {ed_time - st_time}s")
        
        if use_early_stopping:
            if isinstance(mrae_value, torch.Tensor):
                # Detach and convert to numpy, then to a Python scalar if mrae_value is a tensor.
                mrae_value = mrae_value.detach().cpu().numpy().item()
            early_stopping(mrae_value, model)
            if early_stopping.early_stop:
                print("Early stopping")
                break
    
    # get best model
    if use_early_stopping:
        model.load_state_dict(torch.load(early_stopping.checkpoint_path))
        
    return model

In [ ]:
def infer_model(model, test_loader):
    model.eval()  # 모델을 평가 모드로 설정
    predictions = []

    with torch.no_grad():  # 그래디언트 계산을 비활성화
        for X_test, y_test, cluster_ids  in test_loader:  # 타겟 데이터는 필요하지 않으므로 무시
            output = model(X_test)
            predictions.append(output.numpy())
            # predictions.append(np.log(output.numpy()))

    return np.concatenate(predictions)

In [ ]:
# 학습 실행
pretrain = False
pretrain_epochs = 5
num_epochs = 100
batch_size = 32
patience = 30
verbose = True
use_early_stopping = False

input_dim = len(x_col) # Model Input Dimenssion
output_dim = len(y_col) # Model Output Dimenssion

results = []
for i in tqdm(range(50)):
    model = tMeNet(input_dim, output_dim).to(device)

    # 모델 초기화
    tmenet = train_model(model, input_dim, pretrain, pretrain_epochs,
                         num_epochs, train_loader, clusters, patience,
                         verbose, device,use_early_stopping)
    
    # 추론 실행
    predictions = infer_model(tmenet, test_loader)
    y_pred = np.exp(predictions)
    # print(predictions)
    
    y_real = test_dataset.targets #np.exp(test_dataset.targets)
    # y_real = np.exp(y_real)
    # print(f"y real : {y_real}")

    mse = mean_squared_error(y_real, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_real, y_pred)
    mrae = mean_relative_absolute_error(y_real, y_pred)
    mrpe = mean_relative_prediction_error(y_real, y_pred)

    print(f"TEST {i}")
    print(f"MSE : {mse}\nRMSE : {rmse}\nMAE : {mae}\nMRAE : {mrae}\nMRPE : {mrpe}\n")
    
    result = {
        'mse' : mse,
        'rmse' : rmse,
        'mae' : mae,
        'mrae' : mrae,
        'mrpe' : mrpe
    }
    
    results.append(result)

In [ ]:
# 각 평가지표별 리스트 초기화
mse_values = []
rmse_values = []
mae_values = []
mrae_values = []
mrpe_values = []

# 각 결과를 리스트에 추가
for result in results:
    mse_values.append(result['mse'])
    rmse_values.append(result['rmse'])
    mae_values.append(result['mae'])
    mrae_values.append(result['mrae'])  # array에서 값을 추출하여 추가
    mrpe_values.append(result['mrpe'])  # array에서 값을 추출하여 추가

# 평균 및 표준편차 계산
metrics = {
    'mse': mse_values,
    'rmse': rmse_values,
    'mae': mae_values,
    'mrae': mrae_values,
    'mrpe': mrpe_values
}

results_summary = {}
for metric, values in metrics.items():
    mean = np.mean(values)
    std = np.std(values)
    results_summary[metric] = {'mean': mean, 'std': std}

# 결과 출력
for metric, summary in results_summary.items():
    print(f"{metric.upper()}: Mean = {summary['mean']}, Std = {summary['std']}")

In [ ]:
import json

# numpy 배열의 첫 번째 값을 반환하는 함수
def convert_to_serializable(obj):
    if isinstance(obj, np.ndarray):
        return float(obj[0])
    if isinstance(obj, np.float32):
        return float(obj)
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

# JSON 파일로 저장
with open('results_pk.json', 'w') as f:
    json.dump(results, f, default=convert_to_serializable)

In [ ]:
'''
pk data
MSE : 3.803(4.045)
MAE : 1.497(0.619)
MRAE : 97679.546(124694.828)
MRPE : 32216.107(155087.609)
'''

'''
pk data
MSE : 2.293(0.144)
MAE : 1.248(0.046)
MRAE : 59452.078(44309.007)
MRPE : 35282.316(65214.460)
'''

In [ ]:
# import torch.onnx

# # 샘플 입력 데이터 생성
# sample_input = torch.randn(1, input_dim)  # input_dim은 모델의 입력 차원

# # 모델을 ONNX 형식으로 변환 및 저장
# torch.onnx.export(model, sample_input, "./t-menet_240318.onnx", export_params=True)